# DRAG coefficient calibration

Sweep the DRAG beta for the error-sensitive `X-Y90` and `Y-X90` sequences.

In [ ]:
from pathlib import Path

import numpy as np

from qblox_lab.config.device import save_device_configuration
from qblox_lab.config.hardware import create_hardware_agent
from qblox_lab.experiments.cal12_drag import DragCalibration

## Run parameters

In [ ]:
HARDWARE_CONFIG = Path(r"..\config\hw_config.json")
DEVICE_CONFIG = Path(r"..\config\dut_config.json")
FLUX_CONFIG = Path(r"..\config\flux_config.json")
OUTPUT_DIR = Path("data")
QUBITS = ["q0"]
BETAS = np.linspace(start=-20e-9, stop=20e-9, num=41)  # DRAG beta is in seconds
REPETITIONS = 100

# Use None for the configured amp180. A lower value leaves waveform headroom for DRAG;
# compensate with the drive attenuation if the physical rotation must be preserved.
DRIVE_AMPLITUDE = 0.2
READOUT_AMPLITUDE = None
DRIVE_OUTPUT_ATTENUATION = None
READOUT_OUTPUT_ATTENUATION = None
READOUT_INPUT_ATTENUATION = None
TIMEOUT = 300
CREATE_DUMMY_CONNECTIONS = False
SIMULATE = False
SIMULATION_OPTIMAL_BETA = None  # None uses rxy.beta from the device
SIMULATION_NOISE = None
SIMULATION_SEED = 7
PLOT_RESULTS = True
SAVE_FITTED_DEVICE = Path(r"..\config\dut_config_upd.json")

## Hardware and experiment

In [ ]:
hardware_agent = create_hardware_agent(
    hardware_configuration=HARDWARE_CONFIG, device_configuration=DEVICE_CONFIG,
    output_dir=OUTPUT_DIR, create_dummy_connections=CREATE_DUMMY_CONNECTIONS,
)
experiment = DragCalibration(hardware_agent=hardware_agent, qubits=QUBITS, flux_config=FLUX_CONFIG)

## Measurement or simulation

In [ ]:
if SIMULATE:
    dataset = experiment.simulated_data(
        betas=BETAS, repetitions=REPETITIONS, optimal_beta=SIMULATION_OPTIMAL_BETA,
        noise=SIMULATION_NOISE, seed=SIMULATION_SEED,
    )
else:
    dataset = experiment.run_measurement(
        betas=BETAS, repetitions=REPETITIONS, drive_amplitude=DRIVE_AMPLITUDE,
        readout_amplitude=READOUT_AMPLITUDE,
        drive_output_attenuation=DRIVE_OUTPUT_ATTENUATION,
        readout_output_attenuation=READOUT_OUTPUT_ATTENUATION,
        readout_input_attenuation=READOUT_INPUT_ATTENUATION, timeout=TIMEOUT,
    )
dataset

## Analysis

In [ ]:
results = experiment.analysis()
for qubit, result in results.items():
    print(f"{qubit}: beta = {result.optimal_beta / 1e-9:.6g} ns" if result.success else f"{qubit}: DRAG fit failed or crossed outside the sweep")
if PLOT_RESULTS:
    experiment.plot()

## Optional: device update

In [ ]:
if SAVE_FITTED_DEVICE is not None:
    experiment.update_device()
    saved_path = save_device_configuration(hardware_agent.quantum_device, SAVE_FITTED_DEVICE)
    print(f"Updated device configuration saved to {saved_path}")